# 04.R3 — ResNet-50 Supervised Training & 2048-d Feature Extraction

**Phase 04 — Document Processing & OCR (ResNet-50 Visual Pipeline)**  
**Objective:** Load `ResNet50_Weights.DEFAULT`, fine-tune the document category classifier on `train` / `val` splits (isolated from `test`) using `AdamW` + `CosineAnnealingLR` + early stopping, and save `best_model.pt`, `last_model.pt`, `training_history.csv`, and `model_metadata.json` under `models/phase_04/resnet50/`.


In [ ]:
import os
import sys
import time
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

MANIFEST_CSV = paths.interim_doc_proc / 'resnet50' / 'resnet50_dataset_manifest.csv'
MODEL_DIR = paths.resnet50_model_dir
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LABEL_MAP_JSON = MODEL_DIR / 'label_mapping.json'
PREP_CFG_JSON = MODEL_DIR / 'preprocessing_config.json'
BEST_CKPT = MODEL_DIR / 'best_model.pt'
LAST_CKPT = MODEL_DIR / 'last_model.pt'
HIST_CSV = MODEL_DIR / 'training_history.csv'
META_JSON = MODEL_DIR / 'model_metadata.json'

if not MANIFEST_CSV.exists() or not LABEL_MAP_JSON.exists():
    raise FileNotFoundError("Run 01_resnet50_dataset_preparation.ipynb first.")

df_manifest = pd.read_csv(MANIFEST_CSV)
label_info = json.loads(LABEL_MAP_JSON.read_text(encoding='utf-8'))
prep_spec = json.loads(PREP_CFG_JSON.read_text(encoding='utf-8')) if PREP_CFG_JSON.exists() else {
    'image_size': [224, 224], 'imagenet_mean': [0.485, 0.456, 0.406], 'imagenet_std': [0.229, 0.224, 0.225]
}

NUM_CLASSES = int(label_info['num_classes'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RESNET_TRAIN_CONFIG = {
    'architecture': 'ResNet-50 (torchvision ResNet50_Weights.DEFAULT)',
    'num_classes': NUM_CLASSES,
    'max_epochs': 12,
    'batch_size': 16,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'early_stopping_patience': 4,
    'num_workers': 0,
    'device': str(DEVICE),
    'seed': RANDOM_SEED,
}

train_tf = transforms.Compose([
    transforms.Resize(tuple(prep_spec['image_size'])),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=prep_spec['imagenet_mean'], std=prep_spec['imagenet_std']),
])
eval_tf = transforms.Compose([
    transforms.Resize(tuple(prep_spec['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=prep_spec['imagenet_mean'], std=prep_spec['imagenet_std']),
])

class DocumentImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root: Path, transform):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        im = ImageOps.exif_transpose(Image.open(self.root / row['relative_path'])).convert('RGB')
        return self.transform(im), int(row['label_id'])

train_ds = DocumentImageDataset(df_manifest[df_manifest['split'] == 'train'], paths.root, train_tf)
val_ds = DocumentImageDataset(df_manifest[df_manifest['split'] == 'val'], paths.root, eval_tf)
train_loader = DataLoader(train_ds, batch_size=RESNET_TRAIN_CONFIG['batch_size'], shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=RESNET_TRAIN_CONFIG['batch_size'], shuffle=False, num_workers=0)

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=RESNET_TRAIN_CONFIG['learning_rate'], weight_decay=RESNET_TRAIN_CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=RESNET_TRAIN_CONFIG['max_epochs'])

history = []
best_val_acc, best_epoch, no_imp = -1.0, 0, 0

for epoch in range(1, RESNET_TRAIN_CONFIG['max_epochs'] + 1):
    t0 = time.time()
    model.train()
    tr_loss, tr_corr, tr_tot = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * imgs.size(0)
        tr_corr += (out.argmax(dim=1) == lbls).sum().item()
        tr_tot += imgs.size(0)
    scheduler.step()

    model.eval()
    va_loss, va_corr, va_tot = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out = model(imgs)
            loss = criterion(out, lbls)
            va_loss += loss.item() * imgs.size(0)
            va_corr += (out.argmax(dim=1) == lbls).sum().item()
            va_tot += imgs.size(0)

    tr_l, tr_a = tr_loss / max(1, tr_tot), tr_corr / max(1, tr_tot)
    va_l, va_a = va_loss / max(1, va_tot), va_corr / max(1, va_tot)
    is_best = va_a > best_val_acc
    if is_best:
        best_val_acc, best_epoch, no_imp = va_a, epoch, 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'val_acc': va_a, 'config': RESNET_TRAIN_CONFIG}, str(BEST_CKPT))
    else:
        no_imp += 1
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'val_acc': va_a}, str(LAST_CKPT))

    history.append({'epoch': epoch, 'train_loss': round(tr_l, 4), 'train_acc': round(tr_a, 4), 'val_loss': round(va_l, 4), 'val_acc': round(va_a, 4), 'lr': scheduler.get_last_lr()[0], 'duration_sec': round(time.time() - t0, 2), 'is_best': is_best})
    print(f"Epoch {epoch:02d}/{RESNET_TRAIN_CONFIG['max_epochs']:02d} | TrLoss={tr_l:.4f} TrAcc={tr_a:.4f} | ValLoss={va_l:.4f} ValAcc={va_a:.4f} | Best={best_val_acc:.4f}")
    if no_imp >= RESNET_TRAIN_CONFIG['early_stopping_patience']:
        print(f"[EARLY STOPPING] No validation accuracy improvement for {no_imp} epochs.")
        break

df_hist = pd.DataFrame(history)
df_hist.to_csv(HIST_CSV, index=False)
with open(META_JSON, 'w', encoding='utf-8') as f:
    json.dump({'config': RESNET_TRAIN_CONFIG, 'best_epoch': best_epoch, 'best_val_acc': round(best_val_acc, 4), 'embedding_dim': 2048}, f, indent=2)
display(df_hist)
